# Chapter 3 — Parametric Approaches (II): Extreme Value

教材 pp. 36–47；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from extreme_value_calculator import *

Gumbel 极限与 POT 指数尾的独立解析校验；pp.38,44–45。

In [2]:
model = GeneralizedExtremeValue(0,1,0)
assert_allclose(model.quantile(.95),-np.log(-np.log(.95)))
pot = PeaksOverThreshold(2,1,0,.1)
assert_allclose(pot.quantile(.99),2+np.log(10))
assert_allclose(pot.expected_shortfall(.99),3+np.log(10))
assert_allclose(tail_extrapolation(2,.1,.99,.5),2*np.sqrt(10))
print(pot.expected_shortfall(.99))

5.302585092994045


教材 Example 3.5：参数保持教材百分数单位；p.39 / PDF 50。

In [3]:
gev = GeneralizedExtremeValue(2,.7,.3)
assert_allclose(gev.parent_quantile(.995,100),2.537,atol=.001)
assert_allclose(gev.parent_quantile(.999,100),4.322,atol=.001)

## 尾参数估计与POT例题；pp.40–45 / PDF51–56

In [4]:
# 精确Pareto顺序数据的Pickands尾指数。
data=pd.DataFrame({'loss':1/np.arange(1,101,dtype=float)**.4})
assert_allclose(pickands_index(data,'loss',5),.4)
assert hill_index(data,'loss',20)>0
maxima=block_maxima(pd.DataFrame({'x':[1,5,2,7],'block':[0,0,1,1]}),'x','block')
assert_allclose(maxima.x,[5,7])
# Example 3.6：百分数单位beta=.8, xi=.15, u=2, Nu/n=.04。
pot=PeaksOverThreshold(2,.8,.15,.04)
assert_allclose(pot.quantile(.995),3.952,atol=.001)
assert_allclose(pot.expected_shortfall(.995),5.238,atol=.001)
# 原文99.9% ES印为17.578；按式(3.23)应为7.578，不复制该印刷错误。
assert_allclose(pot.quantile(.999),5.942,atol=.001)
assert_allclose(pot.expected_shortfall(.999),7.578,atol=.001)
# MLE对固定分位网格样本拟合，验证参数化符号与概率方向。
from scipy.stats import genextreme
sample=pd.DataFrame({'x':genextreme.ppf((np.arange(1000)+.5)/1000,-.2,loc=2,scale=.7)})
fit=GeneralizedExtremeValue.fit(sample,'x')
assert_allclose([fit.location,fit.scale,fit.shape],[2,.7,.2],atol=.02)
# p.40的Gumbel排序回归，在精确网格样本上恢复参数。
sample=pd.DataFrame({'x':genextreme.ppf(np.arange(1,501)/501,0,loc=2,scale=.7)})
gumbel=GeneralizedExtremeValue.fit_gumbel_regression(sample,'x')
assert_allclose([gumbel.location,gumbel.scale],[2,.7])
# p.45的GPD MLE和矩估计，在指数超额网格上回归。
sample=pd.DataFrame({'x':2-np.log1p(-(np.arange(2000)+.5)/2000)})
pot_fit=PeaksOverThreshold.fit(sample,'x',2)
moment_fit=PeaksOverThreshold.fit_moments(sample,'x',2)
assert_allclose([pot_fit.scale,pot_fit.shape],[1,0],atol=.02)
assert_allclose([moment_fit.scale,moment_fit.shape],[1,0],atol=.02)